In [4]:
pip install ultralytics torch torchvision timm pygame --break-system-packages


Note: you may need to restart the kernel to use updated packages.


In [9]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import pygame

# ── Models ────────────────────────────────────────────────────────────────────
yolo  = YOLO("yolov8n.pt")
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform
midas.eval()

# ── Audio beeps ────────────────────────────────────────────────────────────────
wave = (np.sin(2 * np.pi * freq * t) * volume * 32767).astype(np.int16)

# stereo (L + R)
wave = np.column_stack((wave, wave))

return pygame.sndarray.make_sound(wave)
def make_beep(freq=880, duration_ms=120, volume=0.6):
    sr = 44100
    n  = int(sr * duration_ms / 1000)
    t  = np.linspace(0, duration_ms / 1000, n, False)
    wave = (np.sin(2 * np.pi * freq * t) * volume * 32767).astype(np.int16)

# FIX: convert to 2D mono
    wave = wave.reshape(-1, 1)
    
    return pygame.sndarray.make_sound(wave)

BEEP = {
    "go_slow":    make_beep(880,  180),
    "caution":    make_beep(1100, 120),
    "stop":       make_beep(1400,  80),
    "pedestrian": make_beep(1600, 150),
}
BEEP_INTERVAL = {"go_slow": 35, "caution": 18, "stop": 6, "pedestrian": 9}
beep_timer = 0

def trigger_beep(zone, fidx):
    global beep_timer
    if zone in BEEP and fidx - beep_timer >= BEEP_INTERVAL[zone]:
        BEEP[zone].play()
        beep_timer = fidx

# ── Lane corridor polygon ──────────────────────────────────────────────────────
# These 4 points define the trapezoid between your two white lines.
# TOP-LEFT, TOP-RIGHT, BOTTOM-RIGHT, BOTTOM-LEFT
# Tune these to match your camera's perspective!
LANE_POLY_RATIO = [
    (0.30, 0.54),   # top-left     (x_ratio, y_ratio of frame)
    (0.70, 0.54),   # top-right
    (0.77, 0.99),   # bottom-right
    (0.23, 0.99),   # bottom-left
]

def get_lane_poly(frame_w, frame_h):
    return np.array([
        [int(x * frame_w), int(y * frame_h)]
        for x, y in LANE_POLY_RATIO
    ], dtype=np.int32)

def point_in_corridor(px, py, poly):
    """Returns True if point (px, py) is inside the lane polygon."""
    return cv2.pointPolygonTest(poly, (float(px), float(py)), False) >= 0

def bbox_in_corridor(x1, y1, x2, y2, poly, threshold=0.35):
    """
    Returns True if at least `threshold` fraction of the bbox
    bottom edge lies inside the corridor polygon.
    We check the bottom edge (feet level) — most reliable for real objects.
    """
    bottom_y = y2
    xs = np.linspace(x1, x2, 10)
    inside = sum(1 for x in xs if point_in_corridor(x, bottom_y, poly))
    return (inside / len(xs)) >= threshold

# ── Zone logic ─────────────────────────────────────────────────────────────────
CALIB = 5.0   # tune: place object at 1m, set CALIB = raw_depth_value * 1.0

def classify_zone(dist_m):
    if dist_m < 0.5:  return "stop"
    if dist_m < 1.2:  return "caution"
    if dist_m < 2.5:  return "go_slow"
    return "clear"

ZONE_COLOR = {
    "stop":       (0,  34, 255),
    "caution":    (0, 204, 255),
    "go_slow":    (0, 200,  68),
    "pedestrian": (0, 102, 255),
    "clear":      (100, 100, 100),
}

# ── Draw lane corridor outline ─────────────────────────────────────────────────
def draw_lane(frame, poly, active_zone):
    color = ZONE_COLOR.get(active_zone, (255, 255, 255))

    # Dim everything outside the corridor
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [poly], 255)
    outside = cv2.bitwise_not(mask)
    dark_overlay = frame.copy()
    dark_overlay[outside > 0] = (dark_overlay[outside > 0] * 0.35).astype(np.uint8)
    cv2.addWeighted(dark_overlay, 0.7, frame, 0.3, 0, frame)

    # Subtle tint inside corridor
    tint = frame.copy()
    cv2.fillPoly(tint, [poly], color)
    cv2.addWeighted(tint, 0.06, frame, 0.94, 0, frame)

    # The two white lane lines (left and right edges of trapezoid)
    cv2.line(frame, tuple(poly[0]), tuple(poly[3]), (255, 255, 255), 3, cv2.LINE_AA)
    cv2.line(frame, tuple(poly[1]), tuple(poly[2]), (255, 255, 255), 3, cv2.LINE_AA)

    # "DETECTION ZONE" label inside at top
    mid_x = (poly[0][0] + poly[1][0]) // 2
    mid_y = poly[0][1] + 18
    cv2.putText(frame, "DETECTION ZONE", (mid_x - 65, mid_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (100, 200, 100), 1, cv2.LINE_AA)

# ── Draw Tesla-style arc lines (inside corridor mask) ─────────────────────────
def draw_arcs(frame, poly, active_zone):
    h, w = frame.shape[:2]

    # Build corridor mask for clipping arcs
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly], 255)

    arc_defs = [
        # (y_ratio, x_margin_ratio, color_bgr, zone_key)
        (0.88, 0.09, (0,  200,  68), "go_slow"),
        (0.91, 0.16, (0,  204, 255), "caution"),
        (0.94, 0.23, (0,   34, 255), "stop"),
    ]

    arc_frame = frame.copy()

    for yr, xr, color, zone in arc_defs:
        x1 = int(w * xr)
        x2 = int(w * (1 - xr))
        y_mid = int(h * yr)
        depth = int(h * 0.05)

        pts = np.array([
            [x, int(y_mid - depth * np.sin(np.pi * (x - x1) / (x2 - x1)))]
            for x in range(x1, x2 + 1, 3)
        ], dtype=np.int32)

        thickness = 3 if zone == active_zone else 2
        cv2.polylines(arc_frame, [pts], False, color, thickness, cv2.LINE_AA)

    # Clip arcs to corridor only
    arc_only = cv2.bitwise_and(arc_frame, arc_frame, mask=mask)
    inv_mask = cv2.bitwise_not(mask)
    bg_only  = cv2.bitwise_and(frame, frame, mask=inv_mask)
    blended  = cv2.add(arc_only, bg_only)
    frame[:] = blended

# ── Draw pedestrian expanding rings ───────────────────────────────────────────
def draw_ped_ring(frame, cx, cy, fidx):
    for i, base_r in enumerate([28, 50, 72]):
        phase = (fidx * 4 + i * 24) % 90
        r = base_r + phase
        alpha = max(0.0, 1.0 - phase / 90)
        ov = frame.copy()
        cv2.circle(ov, (cx, cy), r, (0, 102, 255), 2, cv2.LINE_AA)
        cv2.addWeighted(ov, alpha * 0.85, frame, 1 - alpha * 0.85, 0, frame)

# ── Draw bbox + badge ──────────────────────────────────────────────────────────
def draw_detection(frame, x1, y1, x2, y2, dist_m, zone, is_ped):
    color = ZONE_COLOR.get("pedestrian" if is_ped else zone, (128, 128, 128))
    tag   = "PERSON" if is_ped else "VEHICLE"
    label = f"{tag}  {dist_m:.1f}m  {zone.upper().replace('_',' ')}"

    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
    (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    cv2.rectangle(frame, (x1, y1 - lh - 10), (x1 + lw + 10, y1), color, -1)
    cv2.putText(frame, label, (x1 + 5, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

    # Guide lines from bbox bottom to bottom of frame
    h = frame.shape[0]
    cv2.line(frame, (x1, h), (x1, y2), color, 1, cv2.LINE_AA)
    cv2.line(frame, (x2, h), (x2, y2), color, 1, cv2.LINE_AA)

# ── Status bar ─────────────────────────────────────────────────────────────────
def draw_status(frame, zone, dist_m, has_ped):
    h, w = frame.shape[:2]
    ov = frame.copy()
    cv2.rectangle(ov, (0, h - 34), (w, h), (0, 0, 0), -1)
    cv2.addWeighted(ov, 0.72, frame, 0.28, 0, frame)

    if has_ped:
        label, color = "PEDESTRIAN AHEAD", ZONE_COLOR["pedestrian"]
    elif zone == "stop":
        label, color = f"STOP  {dist_m:.1f}m", ZONE_COLOR["stop"]
    elif zone == "caution":
        label, color = f"CAUTION  {dist_m:.1f}m", ZONE_COLOR["caution"]
    elif zone == "go_slow":
        label, color = f"GO SLOW  {dist_m:.1f}m", ZONE_COLOR["go_slow"]
    else:
        label, color = "CLEAR", (80, 80, 80)

    (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.58, 2)
    cx = (w - lw) // 2
    cv2.rectangle(frame, (cx - 10, h - 28), (cx + lw + 10, h - 6), color, -1, cv2.LINE_AA)
    cv2.putText(frame, label, (cx, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, cv2.LINE_AA)

# ── MAIN ───────────────────────────────────────────────────────────────────────
def run(source):
    cap = cv2.VideoCapture(source)
    fidx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            continue

        h, w = frame.shape[:2]
        poly = get_lane_poly(w, h)

        # ── Depth map ──────────────────────────────────────────────────────
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        inp = midas_transforms(rgb)
        with torch.no_grad():
            depth_map = midas(inp).squeeze().numpy()
        depth_map = cv2.resize(depth_map, (w, h))

        # ── YOLO detections ────────────────────────────────────────────────
        results = yolo(frame, verbose=False)[0]

        active_zone  = "clear"
        closest_dist = 9999.0
        has_ped      = False

        # ── Draw base scene ────────────────────────────────────────────────
        draw_lane(frame, poly, "clear")   # first pass, no active zone yet

        # ── Filter detections to corridor only ─────────────────────────────
        detections_in_lane = []

        for box in results.boxes:
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            if conf < 0.45:
                continue
            if cls not in [0, 2, 5, 7]:   # person, car, bus, truck
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # ── KEY FILTER: skip if bbox bottom is outside corridor ────────
            if not bbox_in_corridor(x1, y1, x2, y2, poly):
                continue   # IGNORED — outside the two white lines

            detections_in_lane.append((cls, conf, x1, y1, x2, y2))

        # ── Process only in-lane detections ───────────────────────────────
        for cls, conf, x1, y1, x2, y2 in detections_in_lane:
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            raw_d  = float(depth_map[cy, cx])
            dist_m = CALIB / (raw_d + 1e-6)

            is_ped = cls == 0
            zone   = classify_zone(dist_m)

            if is_ped:
                has_ped = True
                draw_ped_ring(frame, cx, cy, fidx)

            if dist_m < closest_dist:
                closest_dist = dist_m
                if not is_ped:
                    active_zone = zone

            draw_detection(frame, x1, y1, x2, y2, dist_m, zone, is_ped)

        # ── Redraw lane + arcs with active zone ───────────────────────────
        draw_lane(frame, poly, active_zone)
        draw_arcs(frame, poly, active_zone)

        # ── Status + beep ─────────────────────────────────────────────────
        draw_status(frame, active_zone, closest_dist, has_ped)
        beep_zone = "pedestrian" if has_ped else active_zone
        if beep_zone != "clear":
            trigger_beep(beep_zone, fidx)

        # ── HUD ───────────────────────────────────────────────────────────
        cv2.putText(frame, "REAR  |  LANE-ONLY DETECTION", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (140, 140, 140), 1, cv2.LINE_AA)

        cv2.imshow("Reverse Parking Sensor", frame)
        fidx += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run("video.mp4")   # or run(0) for live webcam

C:\Users\soham\anaconda3\envs\finalproject\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.9.23)
Hello from the pygame community. https://www.pygame.org/contribute.html


Using cache found in C:\Users\soham/.cache\torch\hub\intel-isl_MiDaS_master


Loading weights:  None


Using cache found in C:\Users\soham/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\soham/.cache\torch\hub\intel-isl_MiDaS_master


NameError: name 'freq' is not defined

In [10]:
##This is the Project into which the frame is done. White ROI Frame

import cv2
import numpy as np
from ultralytics import YOLO
import time
import winsound


# Try importing winsound (Windows only)
try:
    SOUND_AVAILABLE = True
except:
    SOUND_AVAILABLE = False

# ==========================================
# CONFIG
# ==========================================
MIN_BOX_AREA = 500
EDGE_THRESHOLD = 5000

last_beep_time = 0

# ==========================================
# ROI
# ==========================================
def get_roi(width, height):
    return np.array([
        [int(width * 0.30), int(height * 0.60)],
        [int(width * 0.70), int(height * 0.60)],
        [int(width * 0.95), height],
        [int(width * 0.05), height]
    ], np.int32)

# ==========================================
# HELPERS
# ==========================================
def is_inside_roi(cx, cy, roi):
    return cv2.pointPolygonTest(roi, (cx, cy), False) >= 0

def get_zone(cy, height):
    if cy < int(height * 0.65):
        return "FAR", 0.8
    elif cy < int(height * 0.75):
        return "MID", 0.5
    elif cy < int(height * 0.85):
        return "NEAR", 0.3
    else:
        return "CRITICAL", 0.1

# ==========================================
# SOUND FUNCTION
# ==========================================
def beep(interval):
    global last_beep_time
    current_time = time.time()

    if current_time - last_beep_time > interval:
        if SOUND_AVAILABLE:
            winsound.Beep(2000, 100)
        else:
            print("BEEP")  # fallback

        last_beep_time = current_time

# ==========================================
# WALL DETECTION
# ==========================================
def detect_wall(frame, roi):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    mask = np.zeros_like(gray)
    cv2.fillPoly(mask, [roi], 255)

    roi_edges = cv2.bitwise_and(edges, edges, mask=mask)
    edge_density = np.sum(roi_edges) / 255

    return edge_density > EDGE_THRESHOLD

# ==========================================
# OBJECT DETECTION
# ==========================================
def detect_objects(frame, model):
    results = model(frame, conf=0.45, verbose=False)
    detections = []

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            area = (x2 - x1) * (y2 - y1)

            if area < MIN_BOX_AREA:
                continue

            cx = (x1 + x2) // 2
            cy = y2

            detections.append((x1, y1, x2, y2, cx, cy))

    return detections

# ==========================================
# MAIN PROCESSING
# ==========================================
def process_frame(frame, model):
    height, width = frame.shape[:2]
    roi = get_roi(width, height)

    output = frame.copy()
    cv2.polylines(output, [roi], True, (255, 255, 255), 2)

    detections = detect_objects(frame, model)
    wall_present = detect_wall(frame, roi)

    object_detected = False
    closest_interval = None

    # -----------------------------
    # OBJECT LOGIC
    # -----------------------------
    for (x1, y1, x2, y2, cx, cy) in detections:
        if not is_inside_roi(cx, cy, roi):
            continue

        object_detected = True

        zone, interval = get_zone(cy, height)

        if closest_interval is None or interval < closest_interval:
            closest_interval = interval

        # Draw
        color = (0, 255, 255)
        if zone == "MID":
            color = (0, 165, 255)
        elif zone in ["NEAR", "CRITICAL"]:
            color = (0, 0, 255)

        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)
        cv2.circle(output, (cx, cy), 5, (0, 255, 0), -1)
        cv2.putText(output, zone, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # -----------------------------
    # FINAL DECISION (PRIORITY)
    # -----------------------------
    if wall_present:
        cv2.putText(output, "WALL DETECTED",
                    (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 0, 255), 3)

        beep(0.1)

    elif object_detected:
        cv2.putText(output, "OBJECT DETECTED",
                    (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 165, 255), 2)

        if closest_interval:
            beep(closest_interval)

    else:
        cv2.putText(output, "CLEAR",
                    (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 255, 0), 2)

    return output

# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    model = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        result = process_frame(frame, model)
        cv2.imshow("FINAL SYSTEM", result)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [8]:
import cv2
import numpy as np
from ultralytics import YOLO
import time
import threading

# Try importing winsound (Windows only)
try:
    import winsound
    SOUND_AVAILABLE = True
except:
    SOUND_AVAILABLE = False

# ==========================================
# CONFIG
# ==========================================
MIN_BOX_AREA = 1000
EDGE_THRESHOLD = 8000  # Wall sensitivity
last_beep_time = 0

# ==========================================
# HELPERS
# ==========================================
def get_roi(width, height):
    return np.array([
        [int(width * 0.30), int(height * 0.60)],
        [int(width * 0.70), int(height * 0.60)],
        [int(width * 0.95), height],
        [int(width * 0.05), height]
    ], np.int32)

def is_inside_roi(cx, cy, roi):
    return cv2.pointPolygonTest(roi, (cx, cy), False) >= 0

def get_zone(cy, height):
    """Divides the screen into 4 distinct beeping zones"""
    if cy < int(height * 0.70):
        return "FAR", 0.8, False
    elif cy < int(height * 0.78):
        return "MID", 0.5, False
    elif cy < int(height * 0.86):
        return "NEAR", 0.2, False
    else:
        return "CRITICAL", 0.05, True

# ==========================================
# MISSING FUNCTION: OBJECT DETECTION
# ==========================================
def detect_objects(frame, model):
    results = model(frame, verbose=False)[0]
    detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        conf = float(box.conf[0])
        if conf > 0.4:  # Confidence threshold
            detections.append((x1, y1, x2, y2, cx, cy))
    return detections

# ==========================================
# FIXED: WALL DETECTION (Merged & Cleaned)
# ==========================================
def detect_wall(frame, roi, detections):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 180)

    # Create a mask for the ROI
    mask = np.zeros_like(gray)
    cv2.fillPoly(mask, [roi], 255)

    # --- THE FIX: Remove pedestrians from the wall detection mask ---
    for (x1, y1, x2, y2, cx, cy) in detections:
        # Fill the object's area with 0 (black) so Canny ignores it
        cv2.rectangle(mask, (x1, y1), (x2, y2), 0, -1) 

    # Now calculate density only on what's left
    roi_edges = cv2.bitwise_and(edges, edges, mask=mask)
    edge_density = np.sum(roi_edges) / 255

    return edge_density > EDGE_THRESHOLD

# ==========================================
# SOUND
# ==========================================
def play_beep(freq, duration):
    if SOUND_AVAILABLE:
        winsound.Beep(freq, duration)

def beep(interval, continuous=False):
    global last_beep_time
    curr = time.time()
    if continuous:
        if curr - last_beep_time > 0.1:
            threading.Thread(target=play_beep, args=(1200, 80)).start()
            last_beep_time = curr
    elif curr - last_beep_time > interval:
        threading.Thread(target=play_beep, args=(1000, 150)).start()
        last_beep_time = curr

# ==========================================
# UI & PROCESSING
# ==========================================
def draw_parking_ui(output, width, height):
    overlay = output.copy()
    y_levels = [int(height*0.60), int(height*0.70), int(height*0.78), int(height*0.86), height]
    colors = [(0,255,0), (0,255,255), (0,165,255), (0,0,255)] # G, Y, O, R

    def get_x(y):
        # Linear interpolation for trapezoid sides
        return int((width*0.30) + (y - height*0.60) * (width*0.05 - width*0.30) / (height - height*0.60)), \
               int((width*0.70) + (y - height*0.60) * (width*0.95 - width*0.70) / (height - height*0.60))

    for i in range(4):
        y1, y2 = y_levels[i], y_levels[i+1]
        l1, r1 = get_x(y1)
        l2, r2 = get_x(y2)
        pts = np.array([[l1,y1], [r1,y1], [r2,y2], [l2,y2]], np.int32)
        cv2.fillPoly(overlay, [pts], colors[i])

    cv2.addWeighted(overlay, 0.2, output, 0.8, 0, output)
    cv2.polylines(output, [get_roi(width, height)], True, (255,255,255), 2)
    return output

def process_frame(frame, model):
    h, w = frame.shape[:2]
    roi = get_roi(w, h)
    output = draw_parking_ui(frame.copy(), w, h)
    
    detections = detect_objects(frame, model)
    wall_present = detect_wall(frame, roi, detections)
    
    # Filter for closest object in ROI
    objs_in_roi = [d for d in detections if is_inside_roi(d[4], d[5], roi)]
    closest_obj = max(objs_in_roi, key=lambda x: x[5]) if objs_in_roi else None

    # Priority 1: Wall (Critical)
    if wall_present:
        cv2.putText(output, "STOP: WALL", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 3)
        beep(0.05, continuous=True)
    
    # Priority 2: Identified Object
    elif closest_obj:
        x1, y1, x2, y2, cx, cy = closest_obj
        zone, interval, is_critical = get_zone(cy, h)
        
        cv2.rectangle(output, (x1, y1), (x2, y2), (255, 255, 255), 2)
        cv2.putText(output, f"OBJ: {zone}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        beep(interval, continuous=is_critical)
        
    else:
        cv2.putText(output, "CLEAR", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 2)

    return output

if __name__ == "__main__":
    model = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(0)
    while True:
        ret, frame = cap.read()
        if not ret: break
        result = process_frame(frame, model)
        cv2.imshow("Parking Sensor Pro", result)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release()
    cv2.destroyAllWindows()

In [6]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import time
import threading

# ==========================
# LOAD MODELS
# ==========================
# ==========================
# DEVICE
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================
# LOAD YOLO
# ==========================
model = YOLO("yolov8n.pt")
model.to(device)

# ==========================
# LOAD MIDAS
# ==========================
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.to(device)   # ✅ NOW CORRECT
midas.eval()

transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform
# ==========================
# SOUND
# ==========================
try:
    import winsound
    SOUND_AVAILABLE = True
except:
    SOUND_AVAILABLE = False

last_beep_time = 0

def play_beep(freq=1000, duration=150):
    if SOUND_AVAILABLE:
        winsound.Beep(freq, duration)

def beep(interval, continuous=False):
    global last_beep_time
    t = time.time()

    if continuous:
        play_beep(1200, 300)
        return

    if t - last_beep_time > interval:
        threading.Thread(target=play_beep).start()
        last_beep_time = t

# ==========================
# ROI
# ==========================
def get_roi(w, h):
    return np.array([
        [int(w*0.30), int(h*0.60)],
        [int(w*0.70), int(h*0.60)],
        [int(w*0.95), h],
        [int(w*0.05), h]
    ], np.int32)

def inside_roi(cx, cy, roi):
    return cv2.pointPolygonTest(roi, (cx, cy), False) >= 0

# ==========================
# DEPTH ESTIMATION
# ==========================
def get_depth_map(frame):
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = transform(img)   # ✅ FIXED (no unsqueeze)

    input_batch = input_batch.to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=frame.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    depth = prediction.cpu().numpy()
    depth = cv2.normalize(depth, None, 0, 1, cv2.NORM_MINMAX)

    return depth

# ==========================
# OBJECT DETECTION
# ==========================
def detect_objects(frame):
    results = model(frame)[0]
    detections = []

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cx = int((x1+x2)/2)
        cy = int((y1+y2)/2)

        detections.append((x1, y1, x2, y2, cx, cy))

    return detections

# ==========================
# WALL DETECTION (DEPTH BASED)
# ==========================
def detect_wall(depth, roi):
    mask = np.zeros_like(depth)
    cv2.fillPoly(mask, [roi], 1)

    roi_depth = depth * mask

    close_pixels = np.sum(roi_depth < 0.4)
    total_pixels = np.sum(mask)

    if total_pixels == 0:
        return False

    ratio = close_pixels / total_pixels

    return ratio > 0.6   # wall = large close region

# ==========================
# ZONE LOGIC
# ==========================
def get_zone(cy, h):
    if cy < int(h*0.65):
        return "FAR", 0.8, False
    elif cy < int(h*0.75):
        return "MID", 0.5, False
    elif cy < int(h*0.85):
        return "NEAR", 0.3, False
    else:
        return "CRITICAL", 0.1, True

# ==========================
# MAIN PROCESS
# ==========================
def process_frame(frame):
    h, w = frame.shape[:2]
    roi = get_roi(w, h)

    output = frame.copy()

    depth = get_depth_map(frame)
    detections = detect_objects(frame)

    wall = detect_wall(depth, roi)

    closest = None
    max_cy = 0

    for (x1,y1,x2,y2,cx,cy) in detections:
        if not inside_roi(cx, cy, roi):
            continue

        if cy > max_cy:
            max_cy = cy
            closest = (x1,y1,x2,y2,cx,cy)

    # =====================
    # DECISION
    # =====================
    if wall:
        cv2.putText(output, "WALL", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)
        beep(0.1, True)

    elif closest:
        x1,y1,x2,y2,cx,cy = closest

        zone, interval, critical = get_zone(cy, h)

        color = {
            "FAR": (0,255,0),
            "MID": (0,255,255),
            "NEAR": (0,165,255),
            "CRITICAL": (0,0,255)
        }[zone]

        cv2.rectangle(output,(x1,y1),(x2,y2),color,3)
        cv2.putText(output, zone, (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        if critical:
            beep(interval, True)
        else:
            beep(interval)

    else:
        cv2.putText(output, "CLEAR", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    # draw ROI
    cv2.polylines(output, [roi], True, (255,255,255), 2)

    return output

# ==========================
# MAIN LOOP
# ==========================
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret or frame is None:
        continue

    out = process_frame(frame)

    cv2.imshow("SMART PARKING", out)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Using cache found in C:\Users\soham/.cache\torch\hub\intel-isl_MiDaS_master


Loading weights:  None


Using cache found in C:\Users\soham/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\soham/.cache\torch\hub\intel-isl_MiDaS_master



0: 480x640 1 person, 1 bottle, 66.6ms
Speed: 122.6ms preprocess, 66.6ms inference, 11.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 bottle, 27.1ms
Speed: 3.1ms preprocess, 27.1ms inference, 6.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 bottle, 15.9ms
Speed: 1.9ms preprocess, 15.9ms inference, 5.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 bottle, 13.3ms
Speed: 1.8ms preprocess, 13.3ms inference, 3.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 bottle, 15.1ms
Speed: 1.9ms preprocess, 15.1ms inference, 2.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 bottle, 16.5ms
Speed: 2.0ms preprocess, 16.5ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 bottle, 17.7ms
Speed: 2.2ms preprocess, 17.7ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 bottle, 19.1ms
Speed: